# 03 - MACE Training
Train the message-passing MACE model with the same training setup as ACE and model-specific architecture choices.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.mace_wrapper import MACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, pin_memory=True)

Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize MACE model (shared training setup, model-specific architecture)
model = MACEWrapper(
    num_elements=120,
    r_cut=5.0,
    num_radial=8,
    l_max=2,
    num_blocks=2,  # 2 layers of message passing
    node_dim=16
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)
print(f"Training on device: {trainer.device}\n")

Training on device: cuda



In [4]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/mace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
mace_test_df = pd.DataFrame([test_metrics])
mace_test_df.to_csv("../data/mace_test_metrics.csv", index=False)

# Save the trained model state
torch.save(model.state_dict(), "../data/mace_model.pth")

metrics_df.head()

Epoch 000 | Time: 24.09s | Train E MAE: 631.75 meV/atom | Train F MAE: 166.82 meV/Å | Val E MAE: 126.09 meV/atom | Val F MAE: 258.72 meV/Å
Epoch 001 | Time: 18.85s | Train E MAE: 524.03 meV/atom | Train F MAE: 213.30 meV/Å | Val E MAE: 342.67 meV/atom | Val F MAE: 168.79 meV/Å
Epoch 002 | Time: 18.55s | Train E MAE: 528.04 meV/atom | Train F MAE: 161.15 meV/Å | Val E MAE: 516.09 meV/atom | Val F MAE: 130.96 meV/Å
Epoch 003 | Time: 18.70s | Train E MAE: 342.69 meV/atom | Train F MAE: 109.51 meV/Å | Val E MAE: 416.03 meV/atom | Val F MAE: 71.46 meV/Å
Epoch 004 | Time: 18.62s | Train E MAE: 238.82 meV/atom | Train F MAE: 56.86 meV/Å | Val E MAE: 2.65 meV/atom | Val F MAE: 43.61 meV/Å
Epoch 005 | Time: 18.65s | Train E MAE: 120.01 meV/atom | Train F MAE: 40.43 meV/Å | Val E MAE: 99.79 meV/atom | Val F MAE: 31.42 meV/Å
Epoch 006 | Time: 18.72s | Train E MAE: 81.81 meV/atom | Train F MAE: 24.17 meV/Å | Val E MAE: 79.27 meV/atom | Val F MAE: 17.97 meV/Å
Epoch 007 | Time: 18.72s | Train E MAE:

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,98.425190,631.751136,166.820442,24.092909,11.551377,126.090325,258.722380
1,1,33.877410,524.025872,213.296637,18.850137,11.944654,342.668235,168.788746
2,2,27.069944,528.042022,161.145970,18.551484,19.720467,516.087562,130.964473
3,3,12.516987,342.692085,109.506737,18.700002,11.866075,416.032761,71.463093
4,4,4.805774,238.818355,56.864081,18.621127,0.298072,2.647735,43.605356


In [5]:
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 417,825
Trainable Parameters: 417,825
